# 01 · Benchmark A — harmonic oscillator in a periodic box

**Scientific question.** Does the periodic split-operator propagator reproduce harmonic-oscillator dynamics, and does it converge at the second order Strang splitting predicts?

**Scope.** Periodic (ring) topology, analytical Hermite reference, Trotter and spatial convergence, observables and an error budget.

**Inputs.** `configs/{PROFILE}.yaml` (loaded below). No other notebook needs to have
been run first: this notebook imports everything it needs from
`src/boundary_aware_dynamics` and holds no state from any other notebook.

**Expected outputs.** Density snapshots, infidelity against time, a Trotter convergence fit, and printed diagnostics. Figures are written only by notebook 05.

**Approximate runtime.** about 30 seconds on the `smoke` profile.

**Method.** Second-order Strang splitting with the FFT kinetic phase. Trotter order is measured against exact diagonalisation of the same discrete Hamiltonian so that spatial error cancels; total error is measured against the continuum Hermite reference.

**Assumptions.** The box is large enough that the periodic wrap is negligible for this state — checked below, not assumed.

**References.** See `references/references.bib` and `docs/SCIENTIFIC_METHOD.md`.

**What this notebook does _not_ establish.** Nothing about hard-wall boundaries. The harmonic benchmark validates the integrator and the Fourier convention, not the boundary argument.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from boundary_aware_dynamics.config import load_config
from boundary_aware_dynamics import plotting

PROFILE = os.environ.get("BAD_PROFILE", "smoke")   # "paper" for manuscript numbers
                                                   # (scripts/execute_notebooks.py sets this)
config = load_config(ROOT / "configs" / f"{PROFILE}.yaml")
plotting.apply_style("preview")
print(f"profile={config.profile}  config_hash={config.config_hash}")

## Run the benchmark

All parameters come from the configuration; nothing is redefined here.

In [ ]:
from boundary_aware_dynamics.workflows import (
    run_benchmark, trotter_convergence_study, grid_convergence_study, fit_convergence_slope)

result = run_benchmark(config, "harmonic")
e = result.final_errors
print(f"N={result.grid.n_grid}  n_qubits={result.grid.n_data_qubits}  r={result.metadata['n_steps']}  "
      f"dt={result.metadata['time_step']:.4f}")
print(f"final infidelity      {e.infidelity:.3e}")
print(f"final L2 state error  {e.l2_state_error:.3e}")
print(f"max norm error        {result.propagation.max_norm_error:.2e}")
print(f"reference             {result.metadata['reference_method']}")
print(f"reference truncation  {result.metadata['reference_diagnostics']}")

## Is the periodic box actually large enough?

The reference is a continuum eigenbasis truncated to a finite box. If probability reaches the box edge, the periodic representation is wrong for this problem too — so this is checked rather than asserted.

In [ ]:
edge = np.array([b["wrap_around_probability"] for b in result.boundary])
print(f"maximum probability within 5% of either box edge: {edge.max():.2e}")
print("box truncation is negligible" if edge.max() < 1e-4 else "BOX TOO SMALL for this state")

In [ ]:
fig = plotting.plot_density_snapshots(
    result.grid.positions, result.times,
    {"reference": result.reference.states, "periodic": result.propagation.states},
    np.unique(np.linspace(0, len(result.times) - 1, 4).astype(int)))
plt.show()

## Trotter convergence

Measured against exact diagonalisation of the **same discrete Hamiltonian**, so grid resolution and basis truncation cancel and the fitted slope is the order of the time integrator alone. The first point is excluded from the fit because it lies in the pre-asymptotic transient; the fitted window is reported with the slope.

In [ ]:
study = trotter_convergence_study(config, "harmonic")
for r, dt, err, inf in zip(study.values, study.step_sizes, study.l2_state_error, study.infidelity):
    print(f"  r={r:5d}  dt={dt:.5f}  L2={err:.3e}  infidelity={inf:.3e}")
print(f"\nstate-error slope {study.fit['slope']:.3f}  R2={study.fit['r_squared']:.5f}  "
      f"fitted over dt in {study.fit['fit_interval_dt']}")
inf_fit = fit_convergence_slope(study.step_sizes, study.infidelity, fit_from=1)
print(f"infidelity  slope {inf_fit['slope']:.3f}  R2={inf_fit['r_squared']:.5f}  (expected ~4)")

In [ ]:
fig = plotting.plot_convergence(study.step_sizes, study.l2_state_error, study.fit, expected_slope=2.0)
plt.show()

## Spatial convergence and observables

In [ ]:
grid_study = grid_convergence_study(config, "harmonic")
for n, err in zip(grid_study.values, grid_study.l2_state_error):
    print(f"  N={n:4d}  L2 state error {err:.3e}")
print("\nnote: this measures TOTAL error against the continuum reference, so it saturates")
print("      once the fixed Trotter step dominates — that floor is physical, not a bug.")

In [ ]:
obs = result.observables
print(f"energy drift (max |E(t)-E(0)|) : {np.abs(obs['energy_drift']).max():.3e}")
print(f"initial total energy           : {obs['total_energy'][0]:.6f}")
print(f"<x> range over the period      : [{obs['position_mean'].min():.3f}, {obs['position_mean'].max():.3f}]")
print(f"<p> range over the period      : [{obs['momentum_mean'].min():.3f}, {obs['momentum_mean'].max():.3f}]")

## Error budget for this benchmark

See `docs/ERROR_BUDGET.md` for the full accounting; the measured contributions are:

In [ ]:
ref = result.metadata["reference_diagnostics"]
print(f"reference truncation (tail weight)   {ref['tail_weight']:.2e}")
print(f"box truncation (edge probability)    {edge.max():.2e}")
print(f"Trotter error at this r (vs discrete) {study.l2_state_error[list(study.values).index(result.metadata['n_steps'])] if result.metadata['n_steps'] in list(study.values) else float('nan'):.3e}")
print(f"total error vs continuum reference   {e.l2_state_error:.3e}")

## Summary

**Main findings.** The periodic split-operator propagator reproduces harmonic dynamics with a fitted state-error slope near 2 and an infidelity slope near 4, both with R² above 0.999. Norm is conserved to machine precision and the energy error is a bounded O(dt²) excursion rather than a secular drift.

**Validation checks performed.** Norm conservation, box-edge probability, Trotter slope against the discrete-Hamiltonian reference, infidelity slope, spatial convergence, energy drift.

**Limitations.** The grid sweep measures total error against a continuum reference and therefore saturates once the fixed Trotter step dominates. The harmonic benchmark says nothing about hard walls.

**Generated files.** None directly; notebook 05 exports the corresponding figures.

**Relationship to the manuscript.** Supplies the second-order convergence evidence and the FFT/QFT convention validation.

**Next.** `02_infinite_well_boundary_comparison.ipynb` turns to hard walls and the central boundary argument.